# Восстановление процесса по журналу событий

Анализ журнала `Case2/students.csv` и восстановление наблюдаемой схемы процесса заявок на закупку: активности, переходы, развилки, возвраты и частоты вариантов.


# Шаг 1. Загрузка и сортировка журнала


In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

CSV_PATH = os.path.join('Case2', 'students.csv')

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

# Приводим типы
df['Timestamp'] = pd.to_datetime(df['Timestamp'], utc=True)
df['EventIndex'] = pd.to_numeric(df['EventIndex'], errors='coerce').astype('Int64')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Сортируем события внутри каждого экземпляра по числовому EventIndex
# (файл глобально отсортирован по времени, строки одной заявки не идут подряд)
df = df.sort_values(['ProcessInstanceId', 'EventIndex'], kind='mergesort').reset_index(drop=True)

print('Строк:', len(df))
print('Экземпляров:', df['ProcessInstanceId'].nunique())
print('Колонки:', list(df.columns))
print()
print('Первые строки:')
print(df.head(8).to_string(index=False))


Строк: 6508
Экземпляров: 1000
Колонки: ['ProcessInstanceId', 'EventId', 'EventIndex', 'ActivityId', 'ActivityName', 'EventType', 'Timestamp', 'TaskId', 'UserId', 'Role', 'Result', 'Amount', 'TemplateVersion']

Первые строки:
                   ProcessInstanceId                              EventId  EventIndex                           ActivityId               ActivityName       EventType                 Timestamp                               TaskId                               UserId       Role Result   Amount  TemplateVersion
009608ae-1567-584a-b67a-c7ec60ec587d 7c943483-eeb8-5003-b402-d83bf7f5e937           1 6f98ac87-3a96-5225-b192-cbdcc3b0a72e         Регистрация заявки process_started 2026-05-06 08:57:00+00:00                                  NaN                                  NaN     system    NaN 93822.87                1
009608ae-1567-584a-b67a-c7ec60ec587d efc1b816-3f61-5851-835f-75e1431b2c58           2 39abc4fb-3090-51ae-85d6-1b8972335c53         Первичная проверка  task

# Шаг 2. Профилирование журнала


In [2]:
print('=== Профилирование журнала ===')
print('Экземпляров:', df['ProcessInstanceId'].nunique(), '| строк:', len(df))
print()
print('EventType:')
print(df['EventType'].value_counts().to_string())
print()
print('Активности (ActivityName -> тип события, роль, частота):')
act = df.groupby(['ActivityName', 'EventType', 'Role']).size().reset_index(name='Count')
print(act.sort_values('Count', ascending=False).to_string(index=False))
print()
print('Result (бизнес-результат):')
print(df['Result'].value_counts(dropna=False).to_string())


=== Профилирование журнала ===
Экземпляров: 1000 | строк: 6508

EventType:
EventType
task_completed       4508
process_started      1000
process_completed    1000

Активности (ActivityName -> тип события, роль, частота):
               ActivityName         EventType        Role  Count
         Первичная проверка    task_completed  specialist   1174
 Согласование руководителем    task_completed     manager   1129
         Регистрация заявки   process_started      system   1000
           Проверка бюджета    task_completed     finance    911
         Исполнение закупки    task_completed procurement    911
           Заявка выполнена process_completed      system    911
Дополнительное согласование    task_completed    director    209
           Доработка заявки    task_completed   requester    174
           Заявка отклонена process_completed      system     63
            Заявка отменена process_completed      system     26

Result (бизнес-результат):
Result
accept       4245
NaN        

# Шаг 3. Маршруты и переходы


In [3]:
# Полный маршрут каждого экземпляра (события уже отсортированы по EventIndex)
routes = df.groupby('ProcessInstanceId', sort=False)['ActivityName'].apply(lambda s: ' > '.join(s))

# Пары непосредственно следующих активностей
df['next_activity'] = df.groupby('ProcessInstanceId')['ActivityName'].shift(-1)
trans = (
    df[df['next_activity'].notna()]
    .groupby(['ActivityName', 'next_activity'])
    .size()
    .reset_index(name='Count')
)
trans.columns = ['SourceActivity', 'TargetActivity', 'Count']
trans = trans.sort_values('Count', ascending=False).reset_index(drop=True)

print('=== Переходы между активностями (Source -> Target) ===')
print(trans.to_string(index=False))


=== Переходы между активностями (Source -> Target) ===
             SourceActivity              TargetActivity  Count
         Первичная проверка  Согласование руководителем   1129
         Регистрация заявки          Первичная проверка   1000
         Исполнение закупки            Заявка выполнена    911
 Согласование руководителем            Проверка бюджета    911
           Проверка бюджета          Исполнение закупки    702
           Проверка бюджета Дополнительное согласование    209
Дополнительное согласование          Исполнение закупки    209
           Доработка заявки          Первичная проверка    174
 Согласование руководителем            Доработка заявки    174
         Первичная проверка            Заявка отклонена     45
 Согласование руководителем             Заявка отменена     26
 Согласование руководителем            Заявка отклонена     18


# Шаг 4. Варианты и частоты


In [4]:
variants = routes.value_counts().reset_index()
variants.columns = ['Route', 'NumberOfCases']
total = variants['NumberOfCases'].sum()
variants['Share'] = variants['NumberOfCases'] / total
variants.insert(0, 'Variant', range(1, len(variants) + 1))
variants = variants[['Variant', 'Route', 'NumberOfCases', 'Share']]

print('Всего экземпляров:', total)
print('Уникальных вариантов:', len(variants))
print()
print('Топ-15 вариантов по частоте:')
print(variants.head(15).to_string(index=False))
print()
print('Суммарная доля топ-10 вариантов:', round(variants['Share'].head(10).sum(), 4))


Всего экземпляров: 1000
Уникальных вариантов: 8

Топ-15 вариантов по частоте:
 Variant                                                                                                                                                                                                                                                                     Route  NumberOfCases  Share
       1                                                                                                                                           Регистрация заявки > Первичная проверка > Согласование руководителем > Проверка бюджета > Исполнение закупки > Заявка выполнена            567  0.567
       2                                                                                                             Регистрация заявки > Первичная проверка > Согласование руководителем > Проверка бюджета > Дополнительное согласование > Исполнение закупки > Заявка выполнена            188  0.188
       3                       

# Шаг 5. Возвраты, развилки и классы случаев


In [5]:
# Возвраты
print('=== Возвраты (Result = return) ===')
returns = df[df['Result'] == 'return']
print('Всего возвратов:', len(returns))
print()
print('Возврат инициируется на этапе:')
print(returns.groupby('ActivityName').size().to_string())
print()

# Повторные выполнения активности (тот же ActivityId, другой TaskId)
print('=== Повторы активности ===')
rep = df.groupby(['ProcessInstanceId', 'ActivityId'])['TaskId'].nunique()
rep = rep[rep > 1]
print('Экземпляров с повтором хотя бы одной активности:', rep.index.get_level_values(0).nunique())
print()

# Развилки: активности с несколькими разными исходами
print('=== Развилки (активность с несколькими исходами) ===')
for a, grp in trans.groupby('SourceActivity'):
    if grp['TargetActivity'].nunique() > 1:
        print(a + ':')
        print(grp[['TargetActivity', 'Count']].to_string(index=False))
        print()

# Класс случаев для «Дополнительное согласование»
print('=== Класс случаев: Дополнительное согласование (director) ===')
has_dir = df.groupby('ProcessInstanceId')['ActivityName'].apply(lambda s: 'Дополнительное согласование' in s.tolist())
amt = df.groupby('ProcessInstanceId')['Amount'].first()
comp = pd.DataFrame({'has_director': has_dir, 'Amount': amt})
print('Распределение суммы по наличию доп. согласования:')
print(comp.groupby('has_director')['Amount'].describe().round(2).to_string())
print()
print('Мин. сумма с доп. согласованием:', round(comp[comp['has_director']]['Amount'].min(), 2))
print('Макс. сумма без доп. согласования:', round(comp[~comp['has_director']]['Amount'].max(), 2))


=== Возвраты (Result = return) ===
Всего возвратов: 174

Возврат инициируется на этапе:
ActivityName
Согласование руководителем    174

=== Повторы активности ===
Экземпляров с повтором хотя бы одной активности: 156

=== Развилки (активность с несколькими исходами) ===
Первичная проверка:
            TargetActivity  Count
Согласование руководителем   1129
          Заявка отклонена     45

Проверка бюджета:
             TargetActivity  Count
         Исполнение закупки    702
Дополнительное согласование    209

Согласование руководителем:
  TargetActivity  Count
Проверка бюджета    911
Доработка заявки    174
 Заявка отменена     26
Заявка отклонена     18

=== Класс случаев: Дополнительное согласование (director) ===
Распределение суммы по наличию доп. согласования:
              count       mean       std        min        25%        50%        75%        max
has_director                                                                                   
False         791.0   81123.30

# Шаг 6. Итоговая схема и CSV-файлы


In [6]:
# Уникальные переходы (без дублей) для student_transitions.csv
out_trans = (
    trans[['SourceActivity', 'TargetActivity']]
    .drop_duplicates()
    .sort_values(['SourceActivity', 'TargetActivity'])
    .reset_index(drop=True)
)
out_trans.to_csv(os.path.join('ResultCase2', 'student_transitions.csv'), index=False, encoding='utf-8')
print('Записано:', os.path.join('ResultCase2', 'student_transitions.csv'), '| переходов:', len(out_trans))

# Таблица полных вариантов
variants.to_csv(os.path.join('ResultCase2', 'variants.csv'), index=False, encoding='utf-8')
print('Записано:', os.path.join('ResultCase2', 'variants.csv'), '| вариантов:', len(variants))


Записано: ResultCase2/student_transitions.csv | переходов: 12
Записано: ResultCase2/variants.csv | вариантов: 8


## Итоговая схема процесса

```mermaid
flowchart LR
    A[Регистрация заявки] --> B[Первичная проверка]
    B -->|accept| C[Согласование руководителем]
    B -->|reject| RJ[Заявка отклонена]
    C -->|accept| D[Проверка бюджета]
    C -->|return| RW[Доработка заявки]
    C -->|reject| RJ
    C -->|cancel| CN[Заявка отменена]
    RW --> B
    D -->|сумма до 150000| EX[Исполнение закупки]
    D -->|сумма свыше 150000| DG[Дополнительное согласование]
    DG --> EX
    EX --> OK[Заявка выполнена]
```

### Легенда (роль на этапе)

| Активность | Роль |
|---|---|
| Регистрация заявки | system (авто) |
| Первичная проверка | specialist |
| Согласование руководителем | manager |
| Доработка заявки | requester |
| Проверка бюджета | finance |
| Дополнительное согласование | director |
| Исполнение закупки | procurement |
| Заявка выполнена / отклонена / отменена | system (авто) |

### Наблюдаемые факты

- **Начало** всегда `Регистрация заявки`; **окончания**: `Заявка выполнена` (911), `Заявка отклонена` (63), `Заявка отменена` (26).
- **Возврат** — цикл `Согласование руководителем → Доработка заявки → Первичная проверка` (174 случая).
- **Развилка по сумме**: заявки с `Amount > 150000` проходят `Дополнительное согласование` (director); остальные идут сразу в исполнение.
- Отклонение возможно на двух этапах: `Первичная проверка` (specialist, 45) и `Согласование руководителем` (manager, 18); отмена — только на `Согласование руководителем` (26).
